# 📘 Notebook 3 — Multilingual Fairness Evaluation (English vs Spanish)
This notebook evaluates performance and fairness across English and Spanish subsets of the MultiFinBen dataset.

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

## Step 1 — Load multilingual dataset (generated from Notebook 2)

In [ ]:
df = pd.read_pickle('MultiFinBen_EN_ES_ready.pkl')
df.head()

## Step 2 — Filter valid embeddings (3072‑dim vectors)

In [ ]:
df = df[df['embedding'].apply(lambda x: isinstance(x, list) and len(x)==3072)]
df = df.reset_index(drop=True)
df.head()

## Step 3 — Encode text labels → numeric IDs

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])
le.classes_

## Step 4 — Create language-specific subsets (EN + ES)

In [ ]:
df_en = df[df['language']=='en']
df_es = df[df['language']=='es']
X_en = np.vstack(df_en['embedding'])
y_en = df_en['label_id']
X_es = np.vstack(df_es['embedding'])
y_es = df_es['label_id']

## Step 5 — Load trained classifier (from Notebook 1)

In [ ]:
model = joblib.load('fusion_model.joblib')
model

## Step 6 — Evaluate performance on English subset

In [ ]:
pred_en = model.predict(X_en)
print('--- ENGLISH PERFORMANCE ---')
print(classification_report(y_en, pred_en, target_names=le.classes_))
f1_en = f1_score(y_en, pred_en, average='macro')
f1_en

## Step 7 — Evaluate performance on Spanish subset

In [ ]:
pred_es = model.predict(X_es)
print('--- SPANISH PERFORMANCE ---')
print(classification_report(y_es, pred_es, target_names=le.classes_))
f1_es = f1_score(y_es, pred_es, average='macro')
f1_es

## Step 8 — Compute Fairness Gap (|F1_en − F1_es|)

In [ ]:
fairness_gap = abs(f1_en - f1_es)
fairness_gap

## Step 9 — Confusion Matrix Comparison (EN vs ES)

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,6))
sns.heatmap(confusion_matrix(y_en, pred_en), annot=True, fmt='d', ax=axes[0])
axes[0].set_title('English Confusion Matrix')
sns.heatmap(confusion_matrix(y_es, pred_es), annot=True, fmt='d', ax=axes[1])
axes[1].set_title('Spanish Confusion Matrix')
plt.show()

## Step 10 — Summary Report

In [ ]:
print('F1 English:', f1_en)
print('F1 Spanish:', f1_es)
print('Fairness gap (|EN - ES|):', fairness_gap)